# Build Boundary Meteostat Dataset


In [1]:
from datetime import date
from pathlib import Path

import meteostat as ms
import numpy as np
import pandas as pd


## Settings


In [2]:
BOUNDS = {
    "min_lat": 24.0,
    "max_lat": 31.5,
    "min_lon": -87.7,
    "max_lon": -79.8,
}

START = date(2018, 1, 1)
END = date(2018, 12, 31)

DATASET_NAME = "florida_test"
OUTPUT_ROOT = Path("WeatherData") / DATASET_NAME
STATION_CSV = Path(f"GraphData/{DATASET_NAME}_stations.csv")
NODE_LIST_CSV = Path(f"GraphData/{DATASET_NAME}_nodeList.csv")
EDGE_LIST_CSV = Path(f"GraphData/{DATASET_NAME}_edgeList.csv")

SEARCH_GRID_SIZE = 6
SEARCH_RADIUS_METERS = 2_000_000
SEARCH_LIMIT_PER_POINT = 5_000
K_NEAREST_EDGES = 3
FILL_K_NEAREST = 3

FEATURE_COLUMNS = ["temp", "rhum", "prcp", "wspd"]
FEATURE_FILE_NAMES = {
    "temp": "temp_data.csv",
    "rhum": "humidity_data.csv",
    "prcp": "rain_data.csv",
    "wspd": "wind_speed_data.csv",
}


## Helper Functions


In [3]:
def discover_stations(bounds):
    lat_values = np.linspace(bounds["min_lat"], bounds["max_lat"], num=SEARCH_GRID_SIZE)
    lon_values = np.linspace(bounds["min_lon"], bounds["max_lon"], num=SEARCH_GRID_SIZE)
    all_stations = []

    for lat in lat_values:
        for lon in lon_values:
            point = ms.Point(float(lat), float(lon))
            try:
                nearby = ms.stations.nearby(
                    point,
                    radius=SEARCH_RADIUS_METERS,
                    limit=SEARCH_LIMIT_PER_POINT,
                )
            except Exception as exc:
                print(f"Skipped search point ({lat:.4f}, {lon:.4f}): {exc}")
                continue

            if nearby is not None and not nearby.empty:
                all_stations.append(nearby.reset_index())

    if not all_stations:
        raise RuntimeError("No Meteostat stations were discovered for these bounds.")

    stations = pd.concat(all_stations, ignore_index=True)
    stations["id"] = stations["id"].astype(str)
    stations["latitude"] = pd.to_numeric(stations["latitude"], errors="coerce")
    stations["longitude"] = pd.to_numeric(stations["longitude"], errors="coerce")

    stations = stations[
        stations["latitude"].between(bounds["min_lat"], bounds["max_lat"])
        & stations["longitude"].between(bounds["min_lon"], bounds["max_lon"])
    ]
    stations = stations.dropna(subset=["latitude", "longitude"])
    stations = stations.drop_duplicates(subset="id").sort_values("id").reset_index(drop=True)

    if stations.empty:
        raise RuntimeError("Stations were found nearby, but none were inside the requested bounds.")

    return stations


def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    return 2 * earth_radius_km * np.arcsin(np.sqrt(a))


def build_graph_tables(stations, k=K_NEAREST_EDGES):
    graph_stations = stations.copy().reset_index(drop=True)
    graph_stations["station_id"] = graph_stations["id"].astype(str)
    graph_stations["id"] = np.arange(1, len(graph_stations) + 1, dtype=int)

    node_columns = ["id", "station_id", "latitude", "longitude"]
    optional_columns = ["elevation", "name", "country", "region", "timezone"]
    node_columns.extend([col for col in optional_columns if col in graph_stations.columns])
    node_data = graph_stations[node_columns].copy()

    station_to_int = dict(zip(node_data["station_id"].astype(str), node_data["id"].astype(int)))
    coords = node_data[["latitude", "longitude"]].to_numpy(dtype=float)
    edge_weights = {}

    for i, (lat, lon) in enumerate(coords):
        distances = haversine_km(lat, lon, coords[:, 0], coords[:, 1])
        neighbor_indices = np.argsort(distances)[1 : k + 1]
        for j in neighbor_indices:
            source = int(node_data.loc[i, "id"])
            target = int(node_data.loc[j, "id"])
            left, right = sorted((source, target))
            edge_weights[(left, right)] = min(float(distances[j]), edge_weights.get((left, right), float("inf")))

    edge_rows = [(source, target, weight) for (source, target), weight in sorted(edge_weights.items())]
    edge_data = pd.DataFrame(edge_rows, columns=["source", "target", "weight"])
    return node_data, edge_data, station_to_int


def fetch_hourly_weather(stations, start, end):
    ms.config.block_large_requests = False
    station_ids = stations["id"].astype(str)
    hourly = ms.hourly(
        station_ids,
        start,
        end,
        parameters=[ms.Parameter.TEMP, ms.Parameter.PRCP, ms.Parameter.WSPD, ms.Parameter.RHUM],
    )
    weather = hourly.fetch()

    if weather.empty:
        raise RuntimeError("Meteostat returned no hourly data for this station set and time period.")

    if "temp" in weather.columns:
        weather["temp"] = weather["temp"] * 9 / 5 + 32
    if "wspd" in weather.columns:
        weather["wspd"] = weather["wspd"] * 0.6213711922

    return weather


def build_station_coord_lookup(stations):
    station_coords = stations.copy()
    station_coords["id"] = station_coords["id"].astype(str)
    station_coords["latitude"] = pd.to_numeric(station_coords["latitude"], errors="coerce")
    station_coords["longitude"] = pd.to_numeric(station_coords["longitude"], errors="coerce")
    station_coords = station_coords.dropna(subset=["latitude", "longitude"])
    station_coords = station_coords[["id", "latitude", "longitude"]].drop_duplicates("id")
    return station_coords.set_index("id")[["latitude", "longitude"]]


def fill_hourly_group_from_nearby(group, coord_lookup, numeric_cols, k=FILL_K_NEAREST):
    filled = group.copy()
    if "station" in filled.index.names:
        station_ids = filled.index.get_level_values("station")
    else:
        station_ids = filled.index.astype(str)

    latitudes = station_ids.map(coord_lookup["latitude"])
    longitudes = station_ids.map(coord_lookup["longitude"])
    coords = np.column_stack([latitudes.to_numpy(dtype=float), longitudes.to_numpy(dtype=float)])

    for col in numeric_cols:
        values = np.array(pd.to_numeric(filled[col], errors="coerce"), dtype=float, copy=True)
        missing_idx = np.where(np.isnan(values))[0]
        known_idx = np.where(~np.isnan(values) & ~np.isnan(coords[:, 0]) & ~np.isnan(coords[:, 1]))[0]

        if len(known_idx) == 0:
            continue

        for i in missing_idx:
            if np.isnan(coords[i, 0]) or np.isnan(coords[i, 1]):
                continue
            dists = np.sqrt((coords[known_idx, 0] - coords[i, 0]) ** 2 + (coords[known_idx, 1] - coords[i, 1]) ** 2)
            nearest = known_idx[np.argsort(dists)[:k]]
            if len(nearest) > 0:
                values[i] = np.nanmean(values[nearest])

        filled[col] = values

    return filled


def fill_hourly_weather(weather, stations):
    coord_lookup = build_station_coord_lookup(stations)
    numeric_cols = weather.select_dtypes(include=[np.number]).columns.tolist()

    if isinstance(weather.index, pd.MultiIndex) and "time" in weather.index.names:
        weather = weather.groupby(level="time", group_keys=False).apply(
            fill_hourly_group_from_nearby,
            coord_lookup=coord_lookup,
            numeric_cols=numeric_cols,
        )
    else:
        weather = fill_hourly_group_from_nearby(weather, coord_lookup, numeric_cols)

    has_station_time_index = isinstance(weather.index, pd.MultiIndex) and {"station", "time"}.issubset(weather.index.names)
    if has_station_time_index and "prcp" in weather.columns:
        weather = weather.sort_index()
        weather["prcp"] = weather["prcp"].groupby(level="station", group_keys=True).apply(
            lambda s: pd.to_numeric(s.droplevel("station"), errors="coerce")
            .sort_index()
            .interpolate(method="time")
            .ffill()
            .bfill()
        )

    return weather


def fill_daily_matrix(daily_matrix, station_coords, k=FILL_K_NEAREST):
    filled = daily_matrix.copy()

    for col in filled.columns:
        values = np.array(pd.to_numeric(filled[col], errors="coerce"), dtype=float, copy=True)
        missing_idx = np.where(np.isnan(values))[0]
        known_idx = np.where(~np.isnan(values) & ~np.isnan(station_coords[:, 0]) & ~np.isnan(station_coords[:, 1]))[0]

        if len(known_idx) == 0:
            continue

        for i in missing_idx:
            if np.isnan(station_coords[i, 0]) or np.isnan(station_coords[i, 1]):
                continue
            dists = np.sqrt(
                (station_coords[known_idx, 0] - station_coords[i, 0]) ** 2
                + (station_coords[known_idx, 1] - station_coords[i, 1]) ** 2
            )
            nearest = known_idx[np.argsort(dists)[:k]]
            if len(nearest) > 0:
                values[i] = np.nanmean(values[nearest])

        filled[col] = values

    filled = filled.apply(
        lambda row: pd.Series(row, index=filled.columns).interpolate(limit_direction="both"),
        axis=1,
    )
    return filled


In [4]:
def export_daily_feature_datasets(weather, stations, station_to_int):
    if not isinstance(weather.index, pd.MultiIndex) or not {"station", "time"}.issubset(weather.index.names):
        raise ValueError("weather must have a MultiIndex with 'station' and 'time' levels.")

    all_station_ids = pd.Index(stations["id"].astype(str).drop_duplicates(), name="station")
    station_lookup = build_station_coord_lookup(stations).reindex(all_station_ids)
    station_coords = station_lookup[["latitude", "longitude"]].to_numpy(dtype=float)

    missing_mapping = sorted(set(all_station_ids) - set(station_to_int))
    if missing_mapping:
        raise ValueError(f"Missing integer IDs for stations: {missing_mapping[:10]}")

    daily_feature_datasets = {}

    for feature in FEATURE_COLUMNS:
        if feature not in weather.columns:
            continue

        feature_df = weather[[feature]].reset_index().copy()
        feature_df["station"] = feature_df["station"].astype(str)
        feature_df["date"] = feature_df["time"].dt.date
        feature_df["hour"] = feature_df["time"].dt.hour
        daily_feature_datasets[feature] = {}

        for date_value, group in feature_df.groupby("date"):
            daily_matrix = (
                group.pivot_table(index="station", columns="hour", values=feature, aggfunc="first")
                .reindex(columns=range(24))
                .reindex(all_station_ids)
                .sort_index()
            )
            daily_matrix.columns = [f"hour_{hour:02d}" for hour in daily_matrix.columns]
            daily_matrix = fill_daily_matrix(daily_matrix, station_coords)

            daily_matrix.index = daily_matrix.index.map(lambda station_id: station_to_int[str(station_id)])
            daily_matrix.index.name = "id"
            daily_matrix = daily_matrix.sort_index()

            daily_feature_datasets[feature][date_value] = daily_matrix

            date_folder = OUTPUT_ROOT / f"{date_value.month}_{date_value.day}_{str(date_value.year)[-2:]}"
            date_folder.mkdir(parents=True, exist_ok=True)
            output_path = date_folder / FEATURE_FILE_NAMES.get(feature, f"{feature}_data.csv")
            daily_matrix.to_csv(output_path, index_label="id")

    return daily_feature_datasets


## Build Dataset


In [5]:
stations = discover_stations(BOUNDS)
stations.to_csv(STATION_CSV, index=False)
print(f"Discovered and saved {len(stations)} stations to {STATION_CSV.resolve()}")

node_data, edge_data, station_to_int = build_graph_tables(stations)
node_data.to_csv(NODE_LIST_CSV, index=False)
edge_data.to_csv(EDGE_LIST_CSV, index=False)
print(f"Saved {len(node_data)} integer-ID nodes to {NODE_LIST_CSV.resolve()}")
print(f"Saved {len(edge_data)} integer-ID edges to {EDGE_LIST_CSV.resolve()}")

display(node_data.head())
display(edge_data.head())


Discovered and saved 128 stations to C:\Users\ke119419\Desktop\EEL6812\FinalProject\EEL6812_Final_Project\FinalScripts\GraphData\florida_test_stations.csv
Saved 128 integer-ID nodes to C:\Users\ke119419\Desktop\EEL6812\FinalProject\EEL6812_Final_Project\FinalScripts\GraphData\florida_test_nodeList.csv
Saved 241 integer-ID edges to C:\Users\ke119419\Desktop\EEL6812\FinalProject\EEL6812_Final_Project\FinalScripts\GraphData\florida_test_edgeList.csv


,id,station_id,latitude,longitude,elevation,name,country,region,timezone
0,1,52GQ6,29.6586,-81.6889,15,Palatka Kay Larkin Field,US,FL,America/New_York
1,2,69052,30.4200,-86.6800,12,Mary Esther,US,FL,America/Chicago
2,3,72201,24.5500,-81.1000,1,Key West Airport,US,FL,America/New_York
3,4,72202,25.7833,-80.3167,4,Miami International Airport,US,FL,America/New_York
4,5,72203,26.6833,-80.1000,6,Palm Beach International,US,FL,America/New_York


,source,target,weight
0,1,37,44.513064
1,1,39,40.337417
2,1,101,47.466942
3,2,15,17.163031
4,2,20,0.484539


In [6]:
weather = fetch_hourly_weather(stations, START, END)
print("Initial missing values:")
display(weather.isna().sum())

weather = fill_hourly_weather(weather, stations)
print("Missing values after hourly fill:")
display(weather.isna().sum())


Initial missing values:


temp     11018
rhum     15117
prcp    399673
wspd      6370
dtype: int64

Missing values after hourly fill:


temp    0
rhum    0
prcp    0
wspd    0
dtype: int64

In [7]:
daily_feature_datasets = export_daily_feature_datasets(weather, stations, station_to_int)

print(f"Saved daily weather data under: {OUTPUT_ROOT.resolve()}")
print(f"Total stations expected in each dataset: {len(station_to_int)}")
for feature, date_dict in daily_feature_datasets.items():
    remaining_missing = sum(int(matrix.isna().sum().sum()) for matrix in date_dict.values())
    print(f"{feature}: {len(date_dict)} daily datasets, remaining missing values = {remaining_missing}")


Saved daily weather data under: C:\Users\ke119419\Desktop\EEL6812\FinalProject\EEL6812_Final_Project\FinalScripts\WeatherData\florida_test
Total stations expected in each dataset: 128
temp: 365 daily datasets, remaining missing values = 0
rhum: 365 daily datasets, remaining missing values = 0
prcp: 365 daily datasets, remaining missing values = 0
wspd: 365 daily datasets, remaining missing values = 0


## Check Output


In [8]:
node_check = pd.read_csv(NODE_LIST_CSV)
edge_check = pd.read_csv(EDGE_LIST_CSV)
sample_day = sorted([path for path in OUTPUT_ROOT.iterdir() if path.is_dir()])[0]
sample_temp = pd.read_csv(sample_day / "temp_data.csv")

assert pd.to_numeric(node_check["id"], errors="coerce").notna().all()
assert pd.to_numeric(edge_check["source"], errors="coerce").notna().all()
assert pd.to_numeric(edge_check["target"], errors="coerce").notna().all()
assert "station_id" in node_check.columns
assert "id" in sample_temp.columns
assert "station" not in sample_temp.columns

print("Format check passed.")
display(sample_temp.head())


Format check passed.


,id,hour_00,hour_01,hour_02,hour_03,hour_04,hour_05,hour_06,hour_07,hour_08,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,1,79.58,79.64,79.58,79.58,79.88,78.98,78.68,78.62,79.04,...,81.98,83.18,85.34,85.34,85.28,83.48,82.58,79.46,78.26,78.86
1,2,80.72,79.94,79.58,79.58,79.16,78.80,79.46,79.58,77.84,...,76.04,75.86,76.16,75.62,76.64,76.58,76.46,77.54,78.02,78.56
2,3,84.92,84.92,86.00,84.92,84.92,84.92,84.92,84.92,84.92,...,87.08,86.00,86.00,87.08,87.08,86.00,86.00,86.00,86.00,84.92
3,4,82.04,82.04,82.04,82.04,82.04,82.04,82.04,82.04,82.04,...,84.92,84.92,87.08,87.08,87.98,89.06,87.08,87.08,86.00,84.02
4,5,82.94,82.04,82.04,82.04,82.04,82.04,82.04,82.04,82.04,...,78.98,82.94,82.94,86.00,84.92,87.08,87.08,87.08,84.02,82.94
